# Structured output

In [1]:
%uv pip install 'langchain[openai]~=1.0' pydantic~=2.12

Using Python 3.13.11 environment at: /github.com/sammyne/langchain-tutorials/.venv
Audited 2 packages in 459ms
Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
from langchain_openai import ChatOpenAI

def new_openai_like(**kwargs) -> ChatOpenAI:
  return ChatOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE_URL"],
    model=os.environ["OPENAI_MODEL"],
    **kwargs
  )

In [3]:
import dotenv


dotenv.load_dotenv()

True

## Provider strategy

In [17]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=new_openai_like(),
    response_format=ContactInfo,  # Auto-selects ProviderStrategy
    debug=True,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

[values] {'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6f0f7cdf-1d9b-4a5b-927d-d725d10bebdf')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 229, 'total_tokens': 356, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_provider': 'openai', 'model_name': 'z-ai/glm4.7', 'system_fingerprint': None, 'id': '990eb79a927849f4bdd41bec9d7ff5e0', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c1cd5-abbe-7011-8072-656127052864-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}, 'id': 'call_5d6a545115e14784863b7d25', 'type': 'tool_call'}], usage_metadata={'input_tokens': 229, 'output_tokens': 127, 'total_tokens': 356, 'input_token_

In [ ]:
# 从消息可以看到 LLM 调用了工具
for v in result['messages']:
  v.pretty_print()

================================ Human Message =================================

Extract contact info from: John Doe, john@example.com, (555) 123-4567
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_5d6a545115e14784863b7d25)
 Call ID: call_5d6a545115e14784863b7d25
  Args:
    name: John Doe
    email: john@example.com
    phone: (555) 123-4567
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [19]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')